# Dose Uncertainty Analysis

Analysis of dose uncertainty based on generated anatomical variants.


In [ ]:
# Import libraries
import numpy as np
import nibabel as nib
import SimpleITK as sitk
import os
import glob
from tqdm import tqdm
import json
from collections import defaultdict
import matplotlib.pyplot as plt

In [ ]:
# Data paths
DATA_DIR = '/net/tscratch/people/plgztabor/ROBUST_PLANNING/DATA'
GENERATED_SAMPLES_DIR = '/net/tscratch/people/plgpiotreksl/generated_samples/images_hu'
CSD_RESULTS_DIR = '/net/tscratch/people/plgpiotreksl/csd_verification_results_fixed'
OUTPUT_DIR = '/net/tscratch/people/plgpiotreksl/dose_uncertainty_results_v2'

# Image parameters
SPACING = (1.171875, 1.171875, 3.0)
ORIGIN = (0.0, 0.0, 0.0)
DIRECTION = (-1.0, 0.0, 0.0, 0.0, -1.0, 0.0, 0.0, 0.0, 1.0)
AFF = np.eye(4)


STRUCTURE_LABELS = [2, 3, 4, 5]
STRUCTURE_NAMES = {
    2: 'rectum',
    3: 'bladder',
    4: 'prostate',
    5: 'femur_heads'
}
DOSE_FLIP_FLAGS = {
    '03': 1,  # Flip Z axis
    '17': 0,  # No change
    '25': 1,
    '27': 1,
    '33': 0,
    '42': 0,
    '59': 1,
    '68': 1,
    '71': 1,
    '76': 1,
    '77': 0
}

# Test patient IDs
test_ids = {'02', '03', '17', '24', '25', '27', '33', '42', '48', '57', '59', '68', '71', '76', '77'}

# Number of samples per patient
NUM_SAMPLES = 10

# Create output directories
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/recalculated_doses', exist_ok=True)  # Cache for transformed doses
os.makedirs(f'{OUTPUT_DIR}/uncertainty_maps', exist_ok=True)     # Mean and std dose maps
os.makedirs(f'{OUTPUT_DIR}/figures', exist_ok=True)              # Visualizations


Używane indeksy sampli (10 z 20): [1, 3, 5, 7, 9, 11, 13, 15, 17, 19]


## Diagnostics - Checking dose data availability

In [ ]:
# Check dose data directory structure
dose_dir = f'{DATA_DIR}/DOSES'

if os.path.exists(dose_dir):
    dose_patients = sorted([d for d in os.listdir(dose_dir) if d.startswith('Patient_')])
    if dose_patients:
        example_patient = dose_patients[0]
        example_path = os.path.join(dose_dir, example_patient)
        dose_files = sorted(glob.glob(f'{example_path}/*.nii.gz'))
        
        print(f"\nExample - {example_patient}:")
        print(f"Number of files: {len(dose_files)}")
        if dose_files:
            for i, f in enumerate(dose_files[:3]):
                print(f"  {i+1}. {os.path.basename(f)}")
else:
    if os.path.exists(DATA_DIR):
        subdirs = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])
        print(f"\nDirectories in {DATA_DIR}:")
        for d in subdirs:
            print(f"  - {d}")

In [ ]:
# Filter patients - only those with all required data
generated_files = glob.glob(f'{GENERATED_SAMPLES_DIR}/Patient_*_sample_*_hu.nii.gz')
available_ids = set([os.path.basename(f).split('_')[1] for f in generated_files])

# Check dose data availability
dose_available_ids = set()
if os.path.exists(f'{DATA_DIR}/DOSES'):
    for pid in test_ids:
        dose_patient_dir = f'{DATA_DIR}/DOSES/Patient_{pid}'
        if os.path.exists(dose_patient_dir):
            dose_files = glob.glob(f'{dose_patient_dir}/Patient_{pid}_fraction_1_.nii.gz')
            if dose_files:
                dose_available_ids.add(pid)

# Patients with all required data
test_patient_ids = sorted(test_ids & available_ids & dose_available_ids)

print(f"Test patients: {len(test_ids)}")
print(f"Patients with generated images: {len(available_ids)}")
print(f"Patients with dose data: {len(dose_available_ids)}")
print(f"\nPatients for analysis ({len(test_patient_ids)}): {test_patient_ids}")

if len(test_patient_ids) == 0:
    print("\nNo patients meeting all criteria!")

## Helper Functions

In [ ]:
def load_nifti_as_sitk(filepath, swap_axes=True):
    """
    Loads a NIfTI file and converts it to a SimpleITK Image.
    
    Args:
        filepath: Path to .nii.gz file
        swap_axes: Whether to swap axes (according to data convention)
    
    Returns:
        sitk.Image with set parameters
    """
    data = nib.load(filepath).get_fdata()
    
    if swap_axes:
        data = data.swapaxes(2, 1).swapaxes(1, 0).swapaxes(2, 1)
    
    sitk_img = sitk.GetImageFromArray(data.astype(np.float32))
    sitk_img.SetOrigin(ORIGIN)
    sitk_img.SetSpacing(SPACING)
    sitk_img.SetDirection(DIRECTION)
    
    return sitk_img


def load_nifti_as_array(filepath, swap_axes=True):
    """
    Loads a NIfTI file as a numpy array.
    """
    data = nib.load(filepath).get_fdata()
    
    if swap_axes:
        data = data.swapaxes(2, 1).swapaxes(1, 0).swapaxes(2, 1)
    
    return data


def extract_structure_mask(structure_data, label):
    """
    Extracts a binary mask for a given structure label.
    
    Args:
        structure_data: 3D array with segmentation
        label: Structure label number
    
    Returns:
        Binary mask as numpy array
    """
    mask = (structure_data == label).astype(np.float32)
    return mask


print("Helper functions loaded.")

## Dose Processing Functions


In [ ]:
def recalculate_dose(dose_planned, hu_planned, hu_generated, dose_threshold=0.01, hu_min_threshold=-500, ratio_limits=(0.8, 1.2)):
    """
    Recalculates dose based on differences in HU values.
    """
    dose_recalculated = dose_planned.copy()
    
    dose_mask = (dose_planned > dose_threshold) & (hu_planned > hu_min_threshold)
    
    if not np.any(dose_mask):
        print("  No voxels meeting criteria (dose > threshold and HU > min_threshold)")
        return dose_recalculated.astype(np.float32)
    
    denominator = 1.0 + hu_planned[dose_mask] / 1000.0
    numerator = 1.0 + hu_generated[dose_mask] / 1000.0
    
    ratio = numerator / denominator
    ratio = np.clip(ratio, ratio_limits[0], ratio_limits[1])
    
    dose_recalculated[dose_mask] = dose_planned[dose_mask] * ratio
    
    return dose_recalculated.astype(np.float32)


def apply_transform_to_dose(dose_array, displacement_field, reference_image):
    """
    Applies displacement field transform to a dose array.
    """
    dose_sitk = sitk.GetImageFromArray(dose_array.astype(np.float32))
    dose_sitk.SetOrigin(ORIGIN)
    dose_sitk.SetSpacing(SPACING)
    dose_sitk.SetDirection(DIRECTION)

    displacement_transform = sitk.DisplacementFieldTransform(displacement_field)
    
    resampler = sitk.ResampleImageFilter()
    resampler.SetReferenceImage(reference_image)
    resampler.SetInterpolator(sitk.sitkLinear)
    resampler.SetDefaultPixelValue(0)
    resampler.SetTransform(displacement_transform)
    
    transformed_dose = resampler.Execute(dose_sitk)
    transformed_dose_array = sitk.GetArrayFromImage(transformed_dose)
    
    return transformed_dose_array


print("Dose processing functions loaded.")

## Main Processing Loop


In [ ]:
def process_patient_dose_uncertainty(patient_id, force_recalculate=False):
    """
    Processes a single patient: computes dose uncertainty.
    """
    print(f"\n{'='*60}")
    print(f"Processing patient: {patient_id}")
    print(f"{'='*60}")
    
    ct_planned_path = f'{DATA_DIR}/CT/Patient_{patient_id}/Patient_{patient_id}_fraction_1_.nii.gz'
    dose_planned_files = glob.glob(f'{DATA_DIR}/DOSES/Patient_{patient_id}/Patient_{patient_id}_fraction_1_*.nii.gz')
    structure_path = f'{DATA_DIR}/STRUCTURES/Patient_{patient_id}/Patient_{patient_id}_fraction_1_.nii.gz'
    
    dose_planned_path = dose_planned_files[0]

    print("Loading baseline data...")
    hu_planned = load_nifti_as_array(ct_planned_path)

    dose_planned_raw = nib.load(dose_planned_path).get_fdata()
    flip_flag = DOSE_FLIP_FLAGS.get(patient_id, 0)
    if flip_flag:
        dose_planned_raw = dose_planned_raw[:, :, ::-1]
    
    dose_planned = dose_planned_raw.swapaxes(2, 1).swapaxes(1, 0).swapaxes(2, 1)
    
    structure_data = load_nifti_as_array(structure_path)
    
    ct_planned_sitk = load_nifti_as_sitk(ct_planned_path)
    
    print(f"  CT shape: {hu_planned.shape}")
    print(f"  HU range: [{hu_planned.min():.0f}, {hu_planned.max():.0f}]")
    print(f"  Dose shape: {dose_planned.shape}")
    print(f"  Dose range: [{dose_planned.min():.2f}, {dose_planned.max():.2f}] Gy")
    print(f"  Flip flag: {flip_flag}")
    
    # Get list of generated samples (use all available)
    sample_files = sorted(glob.glob(f'{GENERATED_SAMPLES_DIR}/Patient_{patient_id}_sample_*_hu.nii.gz'))
    
    print(f"\nFound {len(sample_files)} generated samples")
    
    if len(sample_files) == 0:
        print("No generated samples found")
        return None
    
    transformed_doses = []
    
    for sample_idx, sample_path in enumerate(tqdm(sample_files, desc="Processing samples")):
        sample_name = os.path.basename(sample_path).replace('.nii.gz', '')
        
        transformed_dose_path = f'{OUTPUT_DIR}/recalculated_doses/transformed_dose_{sample_name}.nii.gz'
        
        try:
            if os.path.exists(transformed_dose_path) and not force_recalculate:
                dose_transformed = load_nifti_as_array(transformed_dose_path, swap_axes=False)
                transformed_doses.append(dose_transformed)
                continue
            
            transform_path = f'{CSD_RESULTS_DIR}/transforms/transform_{sample_name}.nii.gz'
            
            if not os.path.exists(transform_path):
                print(f"\n  Missing transform for sample {sample_name}")
                continue
            
            hu_generated = load_nifti_as_array(sample_path)
            
            displacement_field = sitk.ReadImage(transform_path)
            displacement_field.SetOrigin(ORIGIN)
            displacement_field.SetSpacing(SPACING)
            displacement_field.SetDirection(DIRECTION)
            
            dose_recalculated = recalculate_dose(dose_planned, hu_planned, hu_generated)

            dose_transformed = apply_transform_to_dose(
                dose_recalculated, 
                displacement_field, 
                ct_planned_sitk
            )
            
            nifti_img = nib.Nifti1Image(dose_transformed, affine=AFF)
            nib.save(nifti_img, transformed_dose_path)
            
            transformed_doses.append(dose_transformed)
            
        except Exception as e:
            print(f"\n  ERROR for sample {sample_name}: {str(e)}")
            import traceback
            traceback.print_exc()
            continue
    
    print(f"\nProcessed {len(transformed_doses)} samples")
    
    if len(transformed_doses) == 0:
        print("No samples processed")
        return None

    results = {}
    
    doses_stack = np.stack(transformed_doses, axis=0)

    dose_mean = np.mean(doses_stack, axis=0)
    dose_std = np.std(doses_stack, axis=0)
    
    nib.save(nib.Nifti1Image(dose_mean, affine=AFF), 
             f'{OUTPUT_DIR}/uncertainty_maps/dose_mean_Patient_{patient_id}.nii.gz')
    nib.save(nib.Nifti1Image(dose_std, affine=AFF), 
             f'{OUTPUT_DIR}/uncertainty_maps/dose_std_Patient_{patient_id}.nii.gz')
    
    print("\nComputing statistics for structures...")
    
    for label in STRUCTURE_LABELS:
        structure_mask = extract_structure_mask(structure_data, label)
        
        if structure_mask.sum() == 0:
            print(f"  Structure {label} ({STRUCTURE_NAMES[label]}) is empty")
            continue
        
        organ_doses = []
        for dose in transformed_doses:
            if dose.shape != structure_mask.shape:
                print(f"  Shape mismatch - dose: {dose.shape}, mask: {structure_mask.shape}")
                continue
            
            mask_bool = structure_mask.astype(bool)
            organ_dose_values = dose[mask_bool]
            organ_doses.append(organ_dose_values)
        
        if len(organ_doses) == 0:
            print(f"  No doses for structure {label}")
            continue

        mean_doses_per_sample = [np.mean(od) for od in organ_doses]
        
        organ_mean_dose = np.mean(mean_doses_per_sample)
        organ_std_dose = np.std(mean_doses_per_sample)
        
        mask_bool = structure_mask.astype(bool)
        organ_uncertainty_map = dose_std[mask_bool]
        organ_mean_uncertainty = np.mean(organ_uncertainty_map)
        
        # Additional statistics: min/max mean dose
        organ_min_mean_dose = np.min(mean_doses_per_sample)
        organ_max_mean_dose = np.max(mean_doses_per_sample)
        
        results[label] = {
            'structure_name': STRUCTURE_NAMES[label],
            'mean_dose': float(organ_mean_dose),
            'std_dose': float(organ_std_dose),
            'mean_uncertainty': float(organ_mean_uncertainty),
            'n_voxels': int(structure_mask.sum()),
            'n_samples': len(organ_doses),
            'dose_range': [float(organ_min_mean_dose), float(organ_max_mean_dose)]
        }
        
        print(f"  {STRUCTURE_NAMES[label]:20s}: "
              f"Mean={organ_mean_dose:.2f} Gy, "
              f"Std={organ_std_dose:.2f} Gy, "
              f"Uncertainty={organ_mean_uncertainty:.2f} Gy, "
              f"Range=[{organ_min_mean_dose:.2f}, {organ_max_mean_dose:.2f}] Gy")
    
    return results


print("Patient processing function loaded.")

## Single Patient Test

Test the algorithm on patient 03 before running for all patients.

In [ ]:
# RESET CACHE - Clear old results before recomputing
print("="*80)
print("Clearing old results")
print("="*80)

# Remove old transformed doses
old_cache_files = glob.glob(f'{OUTPUT_DIR}/recalculated_doses/transformed_dose_*.nii.gz')
if old_cache_files:
    print(f"\nRemoving {len(old_cache_files)} cached files (transformed doses)...")
    for f in old_cache_files:
        os.remove(f)
    print("Old transformed doses removed")
else:
    print("\nNo cache files to remove")

# Remove old uncertainty maps
uncertainty_maps = glob.glob(f'{OUTPUT_DIR}/uncertainty_maps/dose_*.nii.gz')
if uncertainty_maps:
    print(f"\nRemoving {len(uncertainty_maps)} uncertainty maps...")
    for f in uncertainty_maps:
        os.remove(f)
    print("Old uncertainty maps removed")

# Remove progress file
completed_file = f'{OUTPUT_DIR}/completed_patients.json'
if os.path.exists(completed_file):
    os.remove(completed_file)
    print("Removed progress file (completed_patients.json)")

# Remove old per-patient results
old_results = glob.glob(f'{OUTPUT_DIR}/patient_*_results.json')
if old_results:
    print(f"\nRemoving {len(old_results)} old result files...")
    for f in old_results:
        os.remove(f)
    print("Old results removed")

# Remove summary_statistics.json
summary_file = f'{OUTPUT_DIR}/summary_statistics.json'
if os.path.exists(summary_file):
    os.remove(summary_file)
    print("Removed summary_statistics.json")

# Remove dose_stats_stable_diffusion.txt
dose_stats_file = f'{OUTPUT_DIR}/dose_stats_stable_diffusion.txt'
if os.path.exists(dose_stats_file):
    os.remove(dose_stats_file)
    print("Removed dose_stats_stable_diffusion.txt")

# KEEPING:
# - Transforms from CSD_RESULTS_DIR (independent of sample count)

transforms_count = len(glob.glob(f'{CSD_RESULTS_DIR}/transforms/transform_*.nii.gz'))

print(f"\nCached files kept (reused):")
print(f"   - CSD transforms: {transforms_count}")

print(f"\nReady to recompute with {NUM_SAMPLES} samples!")
print("="*80)

## Process All Patients

In [ ]:
# Process all patients with corrected dose orientation
PATIENTS_TO_PROCESS = sorted(DOSE_FLIP_FLAGS.keys())

all_results = {}
completed_patients = []

# Check which patients have already been processed
completed_file = f'{OUTPUT_DIR}/completed_patients.json'
if os.path.exists(completed_file):
    with open(completed_file, 'r') as f:
        completed_patients = json.load(f)
    print(f"Found {len(completed_patients)} already processed patients: {completed_patients}")

print(f"\nPatients for analysis (with corrected orientation): {PATIENTS_TO_PROCESS}")
print(f"   Number of patients: {len(PATIENTS_TO_PROCESS)}")
print(f"\nOrientation correction applied for patients requiring flip:")
print(f"   {[pid for pid, flag in DOSE_FLIP_FLAGS.items() if flag == 1]}")

for patient_id in sorted(PATIENTS_TO_PROCESS):
    if patient_id in completed_patients:
        print(f"\n[{patient_id}] Patient already processed - loading from cache...")
        
        # Load results from cache
        results_file = f'{OUTPUT_DIR}/patient_{patient_id}_results.json'
        if os.path.exists(results_file):
            with open(results_file, 'r') as f:
                all_results[patient_id] = json.load(f)
        continue
    
    try:
        results = process_patient_dose_uncertainty(patient_id, force_recalculate=False)
        
        if results is not None:
            all_results[patient_id] = results
            
            # Save per-patient results
            with open(f'{OUTPUT_DIR}/patient_{patient_id}_results.json', 'w') as f:
                json.dump(results, f, indent=2)
            
            completed_patients.append(patient_id)
            
            # Save progress
            with open(completed_file, 'w') as f:
                json.dump(completed_patients, f, indent=2)
                
    except Exception as e:
        print(f"\nERROR for patient {patient_id}: {str(e)}")
        import traceback
        traceback.print_exc()
        continue

print(f"\n{'='*60}")
print(f"PROCESSING SUMMARY")
print(f"{'='*60}")
print(f"Successfully processed: {len(completed_patients)} patients")
print(f"Patients: {sorted(completed_patients)}")


## Results Aggregation and Analysis

In [ ]:
# Aggregate results for all patients
aggregated_results = {label: {
    'mean_doses': [],
    'std_doses': [],
    'uncertainties': [],
    'n_samples': []
} for label in STRUCTURE_LABELS}

for patient_id, patient_results in all_results.items():
    for label, struct_results in patient_results.items():
        label_int = int(label) if isinstance(label, str) else label
        if label_int in aggregated_results:
            aggregated_results[label_int]['mean_doses'].append(struct_results['mean_dose'])
            aggregated_results[label_int]['std_doses'].append(struct_results['std_dose'])
            aggregated_results[label_int]['uncertainties'].append(struct_results['mean_uncertainty'])
            if 'n_samples' in struct_results:
                aggregated_results[label_int]['n_samples'].append(struct_results['n_samples'])

# Compute summary statistics
summary_stats = {}
for label in STRUCTURE_LABELS:
    if aggregated_results[label]['mean_doses']:
        summary_stats[label] = {
            'structure_name': STRUCTURE_NAMES[label],
            'mean_dose_avg': float(np.mean(aggregated_results[label]['mean_doses'])),
            'mean_dose_std': float(np.std(aggregated_results[label]['mean_doses'])),
            'std_dose_avg': float(np.mean(aggregated_results[label]['std_doses'])),
            'std_dose_std': float(np.std(aggregated_results[label]['std_doses'])),
            'uncertainty_avg': float(np.mean(aggregated_results[label]['uncertainties'])),
            'uncertainty_std': float(np.std(aggregated_results[label]['uncertainties'])),
            'n_patients': len(aggregated_results[label]['mean_doses']),
            'avg_samples_per_patient': float(np.mean(aggregated_results[label]['n_samples'])) if aggregated_results[label]['n_samples'] else 0
        }

# Save summary
with open(f'{OUTPUT_DIR}/summary_statistics.json', 'w') as f:
    json.dump(summary_stats, f, indent=2)

print("\n" + "="*80)
print("RESULTS SUMMARY - Dose Uncertainty Analysis")
print("="*80)
print(f"\nNumber of patients: {len(all_results)}")
print("\nAlgorithm:")
print("  1. Dose recalculation: D_gen = D_plan * (1+HU_gen/1000)/(1+HU_plan/1000)")
print("  2. Transform recalculated dose to first fraction space")
print("  3. Compute mean and std of transformed doses")

print(f"\n{'Structure':<20} {'Mean Dose (Gy)':<20} {'Dose Std (Gy)':<20} {'Uncertainty (Gy)':<20}")
print("-"*80)

for label in STRUCTURE_LABELS:
    if label in summary_stats:
        stats = summary_stats[label]
        print(f"{STRUCTURE_NAMES[label]:<20} "
              f"{stats['mean_dose_avg']:.2f} +/- {stats['mean_dose_std']:.2f}         "
              f"{stats['std_dose_avg']:.2f} +/- {stats['std_dose_std']:.2f}         "
              f"{stats['uncertainty_avg']:.2f} +/- {stats['uncertainty_std']:.2f}")

print("="*80)
print(f"\nResults saved to: {OUTPUT_DIR}")

## Results Visualization

In [ ]:
# Comparative uncertainty plot for structures
if summary_stats:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    structure_names = [STRUCTURE_NAMES[label] for label in STRUCTURE_LABELS if label in summary_stats]
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']
    
    # 1. Mean dose
    mean_doses = [summary_stats[label]['mean_dose_avg'] for label in STRUCTURE_LABELS if label in summary_stats]
    mean_doses_std = [summary_stats[label]['mean_dose_std'] for label in STRUCTURE_LABELS if label in summary_stats]
    
    axes[0].bar(structure_names, mean_doses, yerr=mean_doses_std, capsize=5, color=colors[:len(structure_names)])
    axes[0].set_ylabel('Mean Dose (Gy)', fontsize=12)
    axes[0].set_title('Average Dose per Structure', fontsize=14, fontweight='bold')
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].grid(axis='y', alpha=0.3)
    
    # 2. Dose variability (std) - inter-sample uncertainty
    std_doses = [summary_stats[label]['std_dose_avg'] for label in STRUCTURE_LABELS if label in summary_stats]
    std_doses_std = [summary_stats[label]['std_dose_std'] for label in STRUCTURE_LABELS if label in summary_stats]
    
    axes[1].bar(structure_names, std_doses, yerr=std_doses_std, capsize=5, color=colors[:len(structure_names)])
    axes[1].set_ylabel('Inter-sample Std (Gy)', fontsize=12)
    axes[1].set_title('Dose Variability Between Samples', fontsize=14, fontweight='bold')
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].grid(axis='y', alpha=0.3)
    
    # 3. Uncertainty - mean voxel-wise uncertainty
    uncertainties = [summary_stats[label]['uncertainty_avg'] for label in STRUCTURE_LABELS if label in summary_stats]
    uncertainties_std = [summary_stats[label]['uncertainty_std'] for label in STRUCTURE_LABELS if label in summary_stats]
    
    axes[2].bar(structure_names, uncertainties, yerr=uncertainties_std, capsize=5, color=colors[:len(structure_names)])
    axes[2].set_ylabel('Mean Voxel Uncertainty (Gy)', fontsize=12)
    axes[2].set_title('Dose Uncertainty per Structure', fontsize=14, fontweight='bold')
    axes[2].tick_params(axis='x', rotation=45)
    axes[2].grid(axis='y', alpha=0.3)
    
    plt.suptitle(f'Dose Uncertainty Analysis - {len(all_results)} Patients\n(Algorithm: dose recalculation + transformation)', 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    
    plt.savefig(f'{OUTPUT_DIR}/figures/dose_uncertainty_summary.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\nPlot saved: {OUTPUT_DIR}/figures/dose_uncertainty_summary.png")
else:
    print("No data for visualization - process patients first")

## Final Summary

In [ ]:
# Final summary
print("\n" + "="*80)
print("DOSE UNCERTAINTY ANALYSIS - FINAL SUMMARY")
print("="*80)

print("-"*80)
print(f"Number of processed patients: {len(all_results)}")
print(f"Output directory: {OUTPUT_DIR}")

if summary_stats:
    print("\n" + "-"*80)
    print("STRUCTURE STATISTICS:")
    print("-"*80)
    
    # Structure with highest uncertainty
    max_uncertainty_label = max(summary_stats.keys(), 
                                key=lambda k: summary_stats[k]['uncertainty_avg'])
    max_uncertainty = summary_stats[max_uncertainty_label]
    
    print(f"\nHighest dose uncertainty:")
    print(f"   {max_uncertainty['structure_name']}: "
          f"{max_uncertainty['uncertainty_avg']:.2f} +/- {max_uncertainty['uncertainty_std']:.2f} Gy")
    
    # Structure with lowest uncertainty
    min_uncertainty_label = min(summary_stats.keys(), 
                                key=lambda k: summary_stats[k]['uncertainty_avg'])
    min_uncertainty = summary_stats[min_uncertainty_label]
    
    print(f"\nLowest dose uncertainty:")
    print(f"   {min_uncertainty['structure_name']}: "
          f"{min_uncertainty['uncertainty_avg']:.2f} +/- {min_uncertainty['uncertainty_std']:.2f} Gy")
    
    # Coefficient of variation
    print(f"\nCoefficient of variation (Std/Mean) per structure:")
    for label in STRUCTURE_LABELS:
        if label in summary_stats:
            stats = summary_stats[label]
            cv = stats['std_dose_avg'] / stats['mean_dose_avg'] if stats['mean_dose_avg'] > 0 else 0
            print(f"   {stats['structure_name']:<20}: {cv:.3f} ({cv*100:.1f}%)")

print("\n" + "="*80)
print("Analysis completed successfully")
print(f"\nOutput files saved to: {OUTPUT_DIR}")
print("  - summary_statistics.json - aggregated statistics")
print("  - patient_XX_results.json - per-patient results")
print("  - uncertainty_maps/ - uncertainty maps (dose_mean, dose_std)")
print("  - recalculated_doses/ - cached transformed doses")
print("  - figures/ - visualizations")
print("="*80)

## Method Comparison: Stable Diffusion vs DDF

Comparison of two methods for generating anatomical variants:
- **Stable Diffusion**: Variants generated with a diffusion model (10 samples)
- **DDF (Deformation Field)**: Variants generated via deformation fields (10 samples)

DDF results are stored in `dose_stats.txt`.

In [ ]:
# Load DDF (Deformation Field) method results from dose_stats.txt
ddf_file = '/net/tscratch/people/plgztabor/ROBUST_PLANNING/CODE/DEFORMATIONS/APPLY_TRANSFORMS/dose_stats.txt'

ddf_results = {}
if os.path.exists(ddf_file):
    with open(ddf_file, 'r') as f:
        lines = f.readlines()
    
    current_patient = None
    section = None
    
    for line in lines:
        line = line.strip()
        if line.startswith('Patient_'):
            current_patient = line.split('_')[1]
            ddf_results[current_patient] = {'planned': {}, 'variants': {}}
        elif 'Planned doses means' in line:
            section = 'planned'
        elif 'Dose means from anatomical variants' in line:
            section = 'variants'
        elif current_patient and section and line:
            parts = line.split()
            if len(parts) >= 2:
                if 'rectum' in line.lower():
                    organ = 'rectum'
                elif 'bladder' in line.lower():
                    organ = 'bladder'
                elif 'prostate' in line.lower():
                    organ = 'prostate'
                elif 'femur' in line.lower():
                    organ = 'femur_heads'
                else:
                    continue
                
                try:
                    value = float(parts[-1])
                    ddf_results[current_patient][section][organ] = value
                except:
                    pass

# Structure label to name mapping (matches both methods)
LABEL_TO_NAME = {
    2: 'rectum',
    3: 'bladder',
    4: 'prostate',
    5: 'femur_heads'
}

# Comparison
print("="*140)
print("METHOD COMPARISON - Stable Diffusion vs DDF (Deformation Field)")
print("="*140)

print(f"{'Patient':<10} {'Structure':<20} {'DDF Planned':>18} {'DDF Variants':>18} {'Stable Diff':>15} {'Difference':>12}")
print("-"*140)

differences = []

for pid in sorted(DOSE_FLIP_FLAGS.keys()):
    if pid in all_results and pid in ddf_results:
        sd_data = all_results[pid]
        ddf_data = ddf_results[pid]
        
        for label, name in LABEL_TO_NAME.items():
            if label in sd_data:
                sd_mean = sd_data[label]['mean_dose']
                
                if name in ddf_data['variants']:
                    ddf_planned = ddf_data['planned'].get(name, 0)
                    ddf_variants = ddf_data['variants'][name]
                    diff = sd_mean - ddf_variants
                    diff_pct = 100 * diff / ddf_variants if ddf_variants > 0 else 0
                    
                    differences.append(abs(diff))
                    
                    print(f"{pid:<10} {name:<20} {ddf_planned:>15.2f} Gy {ddf_variants:>15.2f} Gy {sd_mean:>12.2f} Gy {diff:>8.2f} ({diff_pct:+.1f}%)")

print("\n" + "="*140)
print("DIFFERENCE STATISTICS:")
print(f"  Mean difference: {np.mean(differences):.3f} Gy")
print(f"  Max difference: {np.max(differences):.3f} Gy")
print(f"  Min difference: {np.min(differences):.3f} Gy")
print(f"  Median difference: {np.median(differences):.3f} Gy")
print("="*140)

In [ ]:
# Comparison visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Prepare scatter data
ddf_values = []
stable_diff_values = []
structure_colors = []
color_map = {'rectum': '#FF6B6B', 'bladder': '#4ECDC4', 'prostate': '#45B7D1', 'femur_heads': '#96CEB4'}

for pid in sorted(DOSE_FLIP_FLAGS.keys()):
    if pid in all_results and pid in ddf_results:
        for label, name in LABEL_TO_NAME.items():
            if label in all_results[pid]:
                if name in ddf_results[pid]['variants']:
                    ddf_values.append(ddf_results[pid]['variants'][name])
                    stable_diff_values.append(all_results[pid][label]['mean_dose'])
                    structure_colors.append(color_map[name])

# 1. Scatter plot - correlation
axes[0].scatter(ddf_values, stable_diff_values, c=structure_colors, alpha=0.6, s=100)

# 1:1 line
max_val = max(max(ddf_values), max(stable_diff_values))
axes[0].plot([0, max_val], [0, max_val], 'k--', alpha=0.5, label='Perfect match (1:1)')

# +/-5% lines
axes[0].plot([0, max_val], [0, max_val*1.05], 'r--', alpha=0.3, linewidth=1)
axes[0].plot([0, max_val], [0, max_val*0.95], 'r--', alpha=0.3, linewidth=1)

axes[0].set_xlabel('DDF Method - Mean Dose (Gy)', fontsize=12)
axes[0].set_ylabel('Stable Diffusion - Mean Dose (Gy)', fontsize=12)
axes[0].set_title('Mean Dose Comparison\n(DDF vs Stable Diffusion)', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

from scipy.stats import pearsonr
r, p_value = pearsonr(ddf_values, stable_diff_values)
axes[0].text(0.05, 0.95, f'R² = {r**2:.4f}\nR = {r:.4f}', 
             transform=axes[0].transAxes, fontsize=11, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# 2. Histogram of differences
differences_all = np.array(stable_diff_values) - np.array(ddf_values)
axes[1].hist(differences_all, bins=20, color='#4ECDC4', alpha=0.7, edgecolor='black')
axes[1].axvline(0, color='red', linestyle='--', linewidth=2, label='Zero difference')
axes[1].axvline(np.mean(differences_all), color='blue', linestyle='-', linewidth=2, label=f'Mean diff: {np.mean(differences_all):.2f} Gy')

axes[1].set_xlabel('Difference (Stable Diffusion - DDF) [Gy]', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title('Distribution of Differences Between Methods', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].legend()

# Statistics on plot
stats_text = f'Mean: {np.mean(differences_all):.2f} Gy\nStd: {np.std(differences_all):.2f} Gy\nMedian: {np.median(differences_all):.2f} Gy'
axes[1].text(0.65, 0.95, stats_text, transform=axes[1].transAxes, fontsize=10,
             verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/figures/comparison_ddf_vs_stable_diffusion.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nPlot saved: {OUTPUT_DIR}/figures/comparison_ddf_vs_stable_diffusion.png")

# Relative difference analysis
relative_diffs = []
for ddf_val, sd_val in zip(ddf_values, stable_diff_values):
    if ddf_val > 1:  # Only for structures with significant dose
        rel_diff = 100 * (sd_val - ddf_val) / ddf_val
        relative_diffs.append(rel_diff)

print("\n" + "="*80)
print("RELATIVE DIFFERENCE ANALYSIS (%):")
print("="*80)
print(f"Mean relative difference: {np.mean(relative_diffs):.1f}%")
print(f"Median relative difference: {np.median(relative_diffs):.1f}%")
print(f"Range: [{np.min(relative_diffs):.1f}%, {np.max(relative_diffs):.1f}%]")
print(f"Std of relative differences: {np.std(relative_diffs):.1f}%")

# How many cases within +/-5%?
within_5pct = sum(1 for d in relative_diffs if abs(d) <= 5)
within_10pct = sum(1 for d in relative_diffs if abs(d) <= 10)
print(f"\nCases within +/-5%: {within_5pct}/{len(relative_diffs)} ({100*within_5pct/len(relative_diffs):.1f}%)")
print(f"Cases within +/-10%: {within_10pct}/{len(relative_diffs)} ({100*within_10pct/len(relative_diffs):.1f}%)")
print("="*80)

## Generate Output File in DDF-compatible Format

Create a text file with results in the same format as the `dose_stats.txt` file generated by the DDF method.

In [ ]:
# Generate output file in DDF-compatible format (dose_stats.txt)
output_file = f'{OUTPUT_DIR}/dose_stats_stable_diffusion.txt'
STRUCTURE_NAME_MAP = {
    2: 'rectum',
    3: 'bladder',
    4: 'prostate',
    5: 'femur heads'
}

print("="*80)
print("Generating output file: dose_stats_stable_diffusion.txt")
print("="*80)

with open(output_file, 'w') as f:
    for patient_id in sorted(DOSE_FLIP_FLAGS.keys()):
        if patient_id not in all_results:
            continue
        
        f.write("%%%%%%%%%%%%%%%%%%%%\n")
        f.write(f"Patient_{patient_id}\n")
        
        # Load planned dose (first fraction) for "Planned doses means"
        dose_planned_files = glob.glob(f'{DATA_DIR}/DOSES/Patient_{patient_id}/Patient_{patient_id}_fraction_1_*.nii.gz')
        if not dose_planned_files:
            print(f"Missing planned dose for patient {patient_id}")
            continue
        
        # Load planned dose
        dose_planned_raw = nib.load(dose_planned_files[0]).get_fdata()
        flip_flag = DOSE_FLIP_FLAGS.get(patient_id, 0)
        if flip_flag:
            dose_planned_raw = dose_planned_raw[:, :, ::-1]
        dose_planned = dose_planned_raw.swapaxes(2, 1).swapaxes(1, 0).swapaxes(2, 1)
        
        # Load structures
        structure_path = f'{DATA_DIR}/STRUCTURES/Patient_{patient_id}/Patient_{patient_id}_fraction_1_.nii.gz'
        if not os.path.exists(structure_path):
            print(f"Missing structures for patient {patient_id}")
            continue
        
        structure_data = load_nifti_as_array(structure_path)
        
        # Section 1: Planned doses means (from original planned dose)
        f.write("\tPlanned doses means\n")
        for label in [2, 3, 4, 5]:
            if label in STRUCTURE_NAME_MAP:
                mask = (structure_data == label)
                if mask.sum() > 0:
                    mean_dose = dose_planned[mask].mean()
                    f.write(f"\t {STRUCTURE_NAME_MAP[label]} {mean_dose}\n")
        
        # Section 2: Dose means from anatomical variants (from our results)
        f.write("\tDose means from anatomical variants\n")
        patient_results = all_results[patient_id]
        for label in [2, 3, 4, 5]:
            if label in patient_results:
                mean_dose = patient_results[label]['mean_dose']
                f.write(f"\t {STRUCTURE_NAME_MAP[label]} {mean_dose}\n")

print(f"\nFile generated: {output_file}")
print(f"\nFormat is compatible with: dose_stats.txt (DDF method)")
print(f"Contains results for {len([p for p in DOSE_FLIP_FLAGS.keys() if p in all_results])} patients")

# Display first example
print("\n" + "="*80)
print("EXAMPLE - First 20 lines of file:")
print("="*80)
with open(output_file, 'r') as f:
    for i, line in enumerate(f):
        if i < 20:
            print(line, end='')
        else:
            break
print("="*80)